# 15 — Reading Order Sorting + Confidence-Gated Extraction

Implements and demonstrates two targeted improvements to the LayoutLMv3 field
extraction pipeline from notebook 13.

| | |
|---|---|
| **Improvement 1** | Reading order sorting — fix token order before LayoutLMv3 |
| **Improvement 2** | Confidence-gated extraction — regex fallback for uncertain fields |
| **Failure mode addressed (1)** | Swapped dates: both dates assigned to `DUE_DATE` when Tesseract returns tokens out of order |
| **Failure mode addressed (2)** | Low-quality/missing fields when model confidence is below threshold |
| **Module** | `src/extraction_improvements.py` |
| **No retraining** | Weights loaded from disk, read-only |
| **No generative AI** | Discriminative token classifier only |

**Workflow:**
1. Run Sections 0 (setup) after every kernel restart
2. Section 1 shows reading order sorting with visual diff
3. Section 2 shows confidence-gated extraction + calibration histogram
4. Section 3 runs both improvements together on 5 test examples
5. Section 4 compares against the baseline (notebook 13 pipeline)

## 0. Setup — run after every kernel restart

In [1]:
import os, sys, time

# Must be set before any transformers/tokenizers import
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

t0 = time.time()
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor
print('transformers imported in', round(time.time() - t0, 2), 'sec')
print('Python:', sys.executable)

/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers imported in 3.83 sec
Python: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/bin/python


In [2]:
import json
import torch
from pathlib import Path
from PIL import Image
from datasets import load_from_disk

PROJECT_ROOT = Path('..').resolve()
DATASET_DIR  = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
CKPT_PATH    = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'
FIGURES_DIR  = PROJECT_ROOT / 'reports' / 'figures'
MAX_LENGTH   = 512

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

with open(DATASET_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

FIELD_ORDER = [
    'INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE',
    'ISSUER_NAME',    'RECIPIENT_NAME', 'TOTAL_AMOUNT'
]
CLEAN_KEY = {
    'INVOICE_NUMBER': 'invoice_number', 'INVOICE_DATE': 'invoice_date',
    'DUE_DATE': 'due_date', 'ISSUER_NAME': 'issuer_name',
    'RECIPIENT_NAME': 'recipient_name', 'TOTAL_AMOUNT': 'total_amount',
}

raw_dataset = load_from_disk(str(DATASET_DIR))

print(f'Device     : {DEVICE}')
print(f'Checkpoint : {CKPT_PATH}')
print(f'Exists     : {CKPT_PATH.exists()}')
print(f'Dataset    : {raw_dataset}')

Device     : mps
Checkpoint : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/layoutlmv3_fatura/best_checkpoint
Exists     : True
Dataset    : DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})


In [3]:
# Load LayoutLMv3 — read only, no retraining
processor = LayoutLMv3Processor.from_pretrained(
    str(CKPT_PATH),
    apply_ocr=False,
    use_fast=True,
    local_files_only=True,
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(CKPT_PATH),
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
)
model.to(DEVICE)
model.eval()

print('Model loaded OK')
print('  tokenizer :', type(processor.tokenizer).__name__)
print('  num labels:', model.config.num_labels)
print('  device    :', DEVICE)

The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100%|██████████| 216/216 [00:00<00:00, 8216.37it/s]


Model loaded OK
  tokenizer : LayoutLMv3Tokenizer
  num labels: 13
  device    : mps


In [4]:
# Import InvoiceCleaner and all improvements from src/
SRC_DIR = str(PROJECT_ROOT / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from invoice_cleaner import InvoiceCleaner
from extraction_improvements import (
    sort_reading_order,
    ocr_image_tesseract,
    get_raw_predictions_with_confidence,
    extract_with_confidence_gating,
    ROW_THRESHOLD,
)

cleaner = InvoiceCleaner()
print('InvoiceCleaner         : OK')
print('sort_reading_order     : OK')
print('confidence functions   : OK')
print(f'ROW_THRESHOLD          : {ROW_THRESHOLD} (normalised units)')

InvoiceCleaner         : OK
sort_reading_order     : OK
confidence functions   : OK
ROW_THRESHOLD          : 15 (normalised units)


## 1. Reading Order Sorting

**Problem:** Tesseract returns OCR tokens in the order it encounters them during
page segmentation, which is not always strict top-to-bottom left-to-right on
complex invoice layouts.  LayoutLMv3's spatial attention relies on token order
aligning with reading order — out-of-order input directly causes the swapped-dates
failure (both dates assigned to `DUE_DATE`).

**Solution:** Cluster tokens into rows by y0 proximity (±15 units in [0,1000]
normalised space), sort rows top-to-bottom, sort tokens within each row
left-to-right.

In [5]:
# Visual diff: before vs after reading-order sort on 3 test examples
SPLIT = 'test'
N_DIFF = 3

for i in range(N_DIFF):
    example = raw_dataset[SPLIT][i]
    words   = example['words']
    bboxes  = example['bboxes']

    sorted_words, sorted_boxes = sort_reading_order(words, bboxes)

    # Build a mapping from each word to its before/after position
    # (use the first occurrence if duplicates exist)
    orig_pos   = {w: idx for idx, w in enumerate(words)}
    sorted_pos = {w: idx for idx, w in enumerate(sorted_words)}

    # Find tokens whose order changed
    changed = []
    for orig_idx, (w, b) in enumerate(zip(words, bboxes)):
        new_idx = sorted_pos.get(w)
        if new_idx is not None and new_idx != orig_idx:
            changed.append((orig_idx, new_idx, w, b))

    print(f"\n{'='*70}")
    print(f"  TEST {i}: {Path(example['image_path']).stem}")
    print(f"  Total tokens: {len(words)}  |  Reordered: {len(changed)}")
    print('='*70)

    if not changed:
        print('  [no change — tokens already in reading order]')
        continue

    print(f"  {'ORIG_IDX':>8}  {'NEW_IDX':>7}  {'DELTA':>5}  "
          f"{'x0':>4} {'y0':>4} {'x1':>4} {'y1':>4}  WORD")
    print(f"  {'-'*8}  {'-'*7}  {'-'*5}  {'-'*4} {'-'*4} {'-'*4} {'-'*4}  {'-'*25}")
    for orig_idx, new_idx, w, b in changed[:20]:  # cap at 20 rows
        delta = new_idx - orig_idx
        x0, y0, x1, y1 = b
        sign  = '+' if delta > 0 else ''
        print(f"  {orig_idx:>8}  {new_idx:>7}  {sign}{delta:>4}  "
              f"{x0:>4} {y0:>4} {x1:>4} {y1:>4}  {repr(w)[:25]}")
    if len(changed) > 20:
        print(f"  ... {len(changed) - 20} more reordered tokens ...")


  TEST 0: Template1_Instance189
  Total tokens: 98  |  Reordered: 98
  ORIG_IDX  NEW_IDX  DELTA    x0   y0   x1   y1  WORD
  --------  -------  -----  ---- ---- ---- ----  -------------------------
         0       47  +  47    16  630  941  737  'table'
         1       19  +  18   557  170  863  269  'Bill'
         2       20  +  18   557  170  863  269  'to'
         3       21  +  18   557  170  863  269  'Shelly'
         4       22  +  18   557  170  863  269  'Rodriguez'
         5       23  +  18   557  170  863  269  '02547'
         6       24  +  18   557  170  863  269  'Ramos'
         7       25  +  18   557  170  863  269  'Bypass'
         8       26  +  18   557  170  863  269  'Suite'
         9       27  +  18   557  170  863  269  '849'
        10       28  +  18   557  170  863  269  'Williamshaven,'
        11       29  +  18   557  170  863  269  'NC'
        12       30  +  18   557  170  863  269  '38767'
        13       31  +  18   557  170  863  269  'US'


In [6]:
# Show first 20 tokens in original order vs sorted order — makes the effect tangible
EXAMPLE_IDX = 0
SPLIT       = 'test'
N_SHOW      = 20

example = raw_dataset[SPLIT][EXAMPLE_IDX]
words   = example['words']
bboxes  = example['bboxes']

sorted_words, sorted_boxes = sort_reading_order(words, bboxes)

print(f"First {N_SHOW} tokens: BEFORE vs AFTER reading-order sort")
print(f"Example: {Path(example['image_path']).stem}")
print()
print(f"  {'#':>3}  {'ORIGINAL':^28}  {'SORTED':^28}")
print(f"  {'---':>3}  {'-'*28}  {'-'*28}")
for j in range(min(N_SHOW, len(words), len(sorted_words))):
    orig   = repr(words[j])[:26]
    srtd   = repr(sorted_words[j])[:26]
    marker = '  ' if orig == srtd else '<<'
    print(f"  {j:>3}  {orig:<28}  {srtd:<28} {marker}")

First 20 tokens: BEFORE vs AFTER reading-order sort
Example: Template1_Instance189

    #            ORIGINAL                       SORTED           
  ---  ----------------------------  ----------------------------
    0  'table'                       'logo'                       <<
    1  'Bill'                        'TAX'                        <<
    2  'to'                          'INVOICE'                    <<
    3  'Shelly'                      'Date'                       <<
    4  'Rodriguez'                   '29-Apr-2012'                <<
    5  '02547'                       'Due'                        <<
    6  'Ramos'                       'Date'                       <<
    7  'Bypass'                      '07-Aug-2010'                <<
    8  'Suite'                       'PO'                         <<
    9  '849'                         'Number'                     <<
   10  'Williamshaven,'              '25'                         <<
   11  'NC'              

## 2. Confidence-Gated Extraction

**Problem:** LayoutLMv3 always outputs a prediction — even when it is uncertain.
Low-confidence predictions are often wrong.  The InvoiceCleaner regex fallback
is more reliable than a low-confidence model prediction for many field types.

**Solution:** Compute per-field mean softmax confidence.  If confidence >=
`model_threshold` (default 0.70), use the model prediction cleaned by
`InvoiceCleaner`.  If confidence < threshold, pass an empty string so
`InvoiceCleaner`'s OCR-word fallback paths activate instead.

The calibration cell below plots a histogram of per-field confidence scores
over 10 test examples to help you choose the right threshold empirically.

In [7]:
# Show raw fields + per-field confidence for one example
SPLIT       = 'test'
EXAMPLE_IDX = 0
THRESHOLD   = 0.70

example = raw_dataset[SPLIT][EXAMPLE_IDX]
image   = Image.open(example['image_path']).convert('RGB')
words   = example['words']
bboxes  = example['bboxes']

# Apply reading order sort first
words, bboxes = sort_reading_order(words, bboxes)

raw_fields, confidences = get_raw_predictions_with_confidence(
    image, words, bboxes, model, processor, DEVICE, id2label, MAX_LENGTH
)

print(f"{'='*70}")
print(f"  {SPLIT.upper()} {EXAMPLE_IDX}: {Path(example['image_path']).stem}")
print(f"  Threshold: {THRESHOLD}")
print('='*70)
print(f"  {'FIELD':<20}  {'CONF':>6}  {'GATE':>7}  RAW MODEL OUTPUT")
print(f"  {'-'*20}  {'-'*6}  {'-'*7}  {'-'*35}")
for field in FIELD_ORDER:
    conf  = confidences.get(field, 0.0)
    raw   = raw_fields.get(field, '—')
    gate  = 'MODEL' if conf >= THRESHOLD else 'REGEX '
    glyph = '  ' if conf >= THRESHOLD else '**'
    raw_d = (raw[:33] + '..') if len(raw) > 35 else raw
    print(f"  {field:<20}  {conf:>6.3f}  {gate:>7}  {raw_d} {glyph}")
print()
print('  ** = below threshold → InvoiceCleaner regex fallback will be used')

  TEST 0: Template1_Instance189
  Threshold: 0.7
  FIELD                   CONF     GATE  RAW MODEL OUTPUT
  --------------------  ------  -------  -----------------------------------
  INVOICE_NUMBER         0.000   REGEX   — **
  INVOICE_DATE           0.998    MODEL  Date 29-Apr-2012   
  DUE_DATE               0.997    MODEL  Due Date 07-Aug-2010   
  ISSUER_NAME            0.000   REGEX   — **
  RECIPIENT_NAME         1.000    MODEL  Bill to Shelly Rodriguez 02547 Ra..   
  TOTAL_AMOUNT           0.998    MODEL  TOTAL 134.41 USD   

  ** = below threshold → InvoiceCleaner regex fallback will be used


In [8]:
# Calibration histogram — run over N_CAL test examples, collect all confidence scores
# Saves figure to reports/figures/confidence_calibration_histogram.png
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

N_CAL     = 10
SPLIT_CAL = 'test'

all_conf_by_field = {field: [] for field in FIELD_ORDER}

print(f'Collecting confidence scores from {N_CAL} {SPLIT_CAL} examples...')
for i in range(N_CAL):
    example = raw_dataset[SPLIT_CAL][i]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    bboxes  = example['bboxes']
    words, bboxes = sort_reading_order(words, bboxes)
    _, confidences = get_raw_predictions_with_confidence(
        image, words, bboxes, model, processor, DEVICE, id2label, MAX_LENGTH
    )
    for field in FIELD_ORDER:
        conf = confidences.get(field, 0.0)
        all_conf_by_field[field].append(conf)
    print(f'  example {i} done')

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    f'Per-Field Confidence Distribution ({N_CAL} test examples)\n'
    f'Dashed line = threshold 0.70',
    fontsize=13
)

for ax, field in zip(axes.flat, FIELD_ORDER):
    scores = all_conf_by_field[field]
    ax.hist(scores, bins=10, range=(0, 1), color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(0.70, color='crimson', linestyle='--', linewidth=1.5, label='threshold=0.70')
    ax.set_title(field, fontsize=10)
    ax.set_xlabel('Confidence', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=7)
    mean_c = sum(scores) / len(scores) if scores else 0
    ax.text(0.05, ax.get_ylim()[1] * 0.85, f'mean={mean_c:.2f}', fontsize=8, color='navy')

plt.tight_layout()

out_path = FIGURES_DIR / 'confidence_calibration_histogram.png'
fig.savefig(out_path, dpi=120, bbox_inches='tight')
plt.close(fig)

print(f'\nHistogram saved to: {out_path}')
print('\nPer-field summary:')
print(f"  {'FIELD':<20}  {'MEAN':>6}  {'MIN':>6}  {'MAX':>6}  {'>=0.70 %':>8}")
print(f"  {'-'*20}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*8}")
for field in FIELD_ORDER:
    s = all_conf_by_field[field]
    if s:
        pct = 100 * sum(1 for c in s if c >= 0.70) / len(s)
        print(f"  {field:<20}  {sum(s)/len(s):>6.3f}  {min(s):>6.3f}  {max(s):>6.3f}  {pct:>7.0f}%")
    else:
        print(f"  {field:<20}  {'n/a':>6}")

  example 0 done
  example 1 done
  example 2 done
  example 3 done
  example 4 done
  example 5 done
  example 6 done
  example 7 done
  example 8 done
  example 9 done

Histogram saved to: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/reports/figures/confidence_calibration_histogram.png

Per-field summary:
  FIELD                   MEAN     MIN     MAX  >=0.70 %
  --------------------  ------  ------  ------  --------
  INVOICE_NUMBER         0.793   0.000   0.998       80%
  INVOICE_DATE           0.875   0.000   0.998       90%
  DUE_DATE               0.495   0.000   0.997       50%
  ISSUER_NAME            0.499   0.000   0.997       50%
  RECIPIENT_NAME         0.597   0.000   1.000       60%
  TOTAL_AMOUNT           0.897   0.000   0.998       90%


## 3. Combined Test — Improvements 1 + 2 on 5 Examples

Shows RAW (model output) / MODEL CONF / GATE / FINAL (after gating + cleaning)
side by side for each field.

In [9]:
N_EXAMPLES = 5
SPLIT      = 'test'
THRESHOLD  = 0.70

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    words   = example['words']
    bboxes  = example['bboxes']

    # Improvement 1: reading order sort
    words, bboxes = sort_reading_order(words, bboxes)

    # Improvement 2: confidence-gated extraction
    result = extract_with_confidence_gating(
        image, words, bboxes, model, processor, DEVICE, id2label, cleaner,
        model_threshold=THRESHOLD, ocr_text=' '.join(words), max_length=MAX_LENGTH
    )
    raw_fields, confidences = get_raw_predictions_with_confidence(
        image, words, bboxes, model, processor, DEVICE, id2label, MAX_LENGTH
    )

    print(f"\n{'='*90}")
    print(f"  {SPLIT.upper()} {i}: {Path(example['image_path']).stem}")
    print('='*90)
    print(f"  {'FIELD':<20}  {'CONF':>6}  {'GATE':>7}  {'RAW MODEL':^30}  FINAL")
    print(f"  {'-'*20}  {'-'*6}  {'-'*7}  {'-'*30}  {'-'*28}")
    for field in FIELD_ORDER:
        conf      = confidences.get(field, 0.0)
        gate      = 'MODEL' if conf >= THRESHOLD else 'REGEX '
        raw_val   = raw_fields.get(field, '—')
        final_val = result.get(CLEAN_KEY[field], '') or '—'
        raw_d     = (raw_val[:28] + '..') if len(raw_val) > 30 else raw_val
        final_d   = (final_val[:26] + '..') if len(final_val) > 28 else final_val
        print(f"  {field:<20}  {conf:>6.3f}  {gate:>7}  {raw_d:<30}  {final_d}")


  TEST 0: Template1_Instance189
  FIELD                   CONF     GATE            RAW MODEL             FINAL
  --------------------  ------  -------  ------------------------------  ----------------------------
  INVOICE_NUMBER         0.000   REGEX   —                               —
  INVOICE_DATE           0.998    MODEL  Date 29-Apr-2012                29-Apr-2012
  DUE_DATE               0.997    MODEL  Due Date 07-Aug-2010            07-Aug-2010
  ISSUER_NAME            0.000   REGEX   —                               —
  RECIPIENT_NAME         1.000    MODEL  Bill to Shelly Rodriguez 025..  Shelly Rodriguez
  TOTAL_AMOUNT           0.998    MODEL  TOTAL 134.41 USD                134.41 USD

  TEST 1: Template38_Instance29
  FIELD                   CONF     GATE            RAW MODEL             FINAL
  --------------------  ------  -------  ------------------------------  ----------------------------
  INVOICE_NUMBER         0.998    MODEL  INVOICE # 9Y1M9d-232            9Y1M9

## 4. Comparison vs Baseline (Notebook 13 Pipeline)

Same 5 examples through the **old pipeline** (no sorting, no gating) versus
the **new pipeline** (reading order sort + confidence gating).

In [10]:
def get_raw_predictions_baseline(image, words, bboxes):
    """Baseline pipeline — identical to notebook 13, no improvements applied."""
    encoding = processor(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    with torch.no_grad():
        outputs = model(**{k: v.to(DEVICE) for k, v in encoding.items()})
    token_preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()
    word_ids    = encoding.word_ids(batch_index=0)
    word_preds  = {}
    for ti, wi in enumerate(word_ids):
        if wi is not None and wi not in word_preds:
            word_preds[wi] = token_preds[ti]
    aligned_words    = [words[i]      for i in sorted(word_preds)]
    aligned_pred_ids = [word_preds[i] for i in sorted(word_preds)]
    fields, current_field, current_tokens = {}, None, []
    for label_id, word in zip(aligned_pred_ids, aligned_words):
        label = id2label[label_id]
        if label == 'O':
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t: fields[current_field] = t
                current_field, current_tokens = None, []
        elif label.startswith('B-'):
            if current_field:
                t = ' '.join(current_tokens).strip()
                if t: fields[current_field] = t
            current_field, current_tokens = label[2:], [word]
        elif label.startswith('I-'):
            fn = label[2:]
            if current_field == fn:
                current_tokens.append(word)
            elif current_field is None and fn in fields:
                fields[fn] += ' ' + word
            elif current_field is None:
                current_field, current_tokens = fn, [word]
            else:
                t = ' '.join(current_tokens).strip()
                if t: fields[current_field] = t
                current_field, current_tokens = fn, [word]
    if current_field:
        t = ' '.join(current_tokens).strip()
        if t: fields[current_field] = t
    return fields

print('Baseline function defined OK')

Baseline function defined OK


In [11]:
N_EXAMPLES = 5
SPLIT      = 'test'
THRESHOLD  = 0.70

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    orig_words  = list(example['words'])
    orig_bboxes = list(example['bboxes'])

    # ── Baseline (notebook 13) ──────────────────────────────────────────
    raw_base    = get_raw_predictions_baseline(image, orig_words, orig_bboxes)
    cleaned_base = cleaner.clean(raw_base, ocr_words=orig_words)

    # ── Improved pipeline ───────────────────────────────────────────────
    sorted_words, sorted_bboxes = sort_reading_order(orig_words, orig_bboxes)
    improved = extract_with_confidence_gating(
        image, sorted_words, sorted_bboxes, model, processor, DEVICE,
        id2label, cleaner, model_threshold=THRESHOLD,
        ocr_text=' '.join(sorted_words), max_length=MAX_LENGTH
    )

    print(f"\n{'='*80}")
    print(f"  {SPLIT.upper()} {i}: {Path(example['image_path']).stem}")
    print('='*80)
    print(f"  {'FIELD':<20}  {'BASELINE':^28}  IMPROVED")
    print(f"  {'-'*20}  {'-'*28}  {'-'*28}")
    changed = 0
    for field in FIELD_ORDER:
        ck      = CLEAN_KEY[field]
        base_v  = cleaned_base.get(ck, '') or '—'
        impr_v  = improved.get(ck, '')    or '—'
        marker  = ' <<' if base_v != impr_v else ''
        base_d  = (base_v[:26] + '..') if len(base_v) > 28 else base_v
        impr_d  = (impr_v[:26] + '..') if len(impr_v) > 28 else impr_v
        print(f"  {field:<20}  {base_d:<28}  {impr_d}{marker}")
        if base_v != impr_v:
            changed += 1
    print(f"  → {changed} field(s) changed")


  TEST 0: Template1_Instance189
  FIELD                           BASELINE            IMPROVED
  --------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER        —                             —
  INVOICE_DATE          29-Apr-2012                   29-Apr-2012
  DUE_DATE              07-Aug-2010                   07-Aug-2010
  ISSUER_NAME           —                             —
  RECIPIENT_NAME        Shelly Rodriguez              Shelly Rodriguez
  TOTAL_AMOUNT          134.41 USD                    134.41 USD
  → 0 field(s) changed

  TEST 1: Template38_Instance29
  FIELD                           BASELINE            IMPROVED
  --------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER        9Y1M9d-232                    9Y1M9d-232
  INVOICE_DATE          15-Feb-1993                   15-Feb-1993
  DUE_DATE              14-Jan-2007                   14-Jan-2007
  ISSUER_NAME           —                 

In [12]:
# --- Notebook 15: test the known-failure real-world files (Section 4 style) ---

from pathlib import Path
from PIL import Image

# Replace ... with your actual project root automatically
REAL_WORLD_IMAGES = [
    PROJECT_ROOT / 'data' / 'external' / 'Template17_Instance182.jpg',  # swapped dates
    PROJECT_ROOT / 'data' / 'external' / 'doc_i_1.webp',                # missed recipient
    PROJECT_ROOT / 'data' / 'external' / 'doc_i_3.webp',                # missed number
    PROJECT_ROOT / 'data' / 'external' / 'invoice-0-4.pdf',             # wrong number (TO)
]

THRESHOLD = 0.70

def load_first_page(path: Path) -> Image.Image:
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        from pdf2image import convert_from_path
        pages = convert_from_path(str(path), first_page=1, last_page=1)
        if not pages:
            raise RuntimeError(f"No pages rendered for PDF: {path}")
        return pages[0].convert("RGB")
    return Image.open(path).convert("RGB")

# Needs Section 4 baseline fn already defined in notebook 15:
# get_raw_predictions_baseline(...)
assert "get_raw_predictions_baseline" in globals(), \
    "Run notebook 15 Section 4 cell that defines get_raw_predictions_baseline() first."

for p in REAL_WORLD_IMAGES:
    p = Path(p)
    print(f"\n{'='*90}")
    print(f"  FILE: {p.name}")
    print("="*90)

    if not p.exists():
        print("  File not found, skipping.")
        continue

    try:
        image = load_first_page(p)

        # OCR (same engine as notebook 15 improvements setup: tesseract)
        orig_words, orig_bboxes = ocr_image_tesseract(image)

        # Baseline (no sort, no confidence gating)
        raw_base = get_raw_predictions_baseline(image, orig_words, orig_bboxes)
        cleaned_base = cleaner.clean(raw_base, ocr_words=orig_words)

        # Improved (sort + confidence gating)
        sorted_words, sorted_bboxes = sort_reading_order(orig_words, orig_bboxes)
        improved = extract_with_confidence_gating(
            image,
            sorted_words,
            sorted_bboxes,
            model,
            processor,
            DEVICE,
            id2label,
            cleaner,
            model_threshold=THRESHOLD,
            ocr_text=' '.join(sorted_words),
            max_length=MAX_LENGTH,
        )

        print(f"  {'FIELD':<20}  {'BASELINE':<30}  IMPROVED")
        print(f"  {'-'*20}  {'-'*30}  {'-'*30}")

        changed = 0
        for field in FIELD_ORDER:
            ck = CLEAN_KEY[field]
            b = cleaned_base.get(ck, '') or '—'
            n = improved.get(ck, '') or '—'
            mark = " <<" if b != n else ""
            b_d = (b[:28] + "..") if len(b) > 30 else b
            n_d = (n[:28] + "..") if len(n) > 30 else n
            print(f"  {field:<20}  {b_d:<30}  {n_d}{mark}")
            changed += int(b != n)

        print(f"  -> {changed} field(s) changed")

    except Exception as e:
        print(f"  Error: {e.__class__.__name__}: {e}")



  FILE: Template17_Instance182.jpg
  File not found, skipping.

  FILE: doc_i_1.webp
  FIELD                 BASELINE                        IMPROVED
  --------------------  ------------------------------  ------------------------------
  INVOICE_NUMBER        12345                           12345
  INVOICE_DATE          —                               —
  DUE_DATE              —                               —
  ISSUER_NAME           Imani Olowe                     Imani Olowe
  RECIPIENT_NAME        Imani Olowe +123-456-7890       Imani Olowe +123-456-7890
  TOTAL_AMOUNT          $2750                           $2750
  -> 0 field(s) changed

  FILE: doc_i_3.webp
  FIELD                 BASELINE                        IMPROVED
  --------------------  ------------------------------  ------------------------------
  INVOICE_NUMBER        —                               —
  INVOICE_DATE          —                               —
  DUE_DATE              —                               —
